# Docking with a ChemGraph AI agent

**Companion to `docking_demo.ipynb`.** That notebook runs the docking pipeline directly.
Here you do the same science by simply **asking an AI agent in plain English** — ChemGraph's
`molecular_docking` workflow picks the right tool, runs **AutoDock Vina**, and returns the
result. We then render the docked poses in **interactive 3D**, right in the notebook.

You can:
- dock any **candidate** (a SMILES, a molecule name, or a PubChem CID) into a target,
- bring your **own receptor** (a PDB ID, a `.pdb` file, or a SMILES),
- **screen several candidates** at once and rank them,
- **see** every result in 3D.


In [ ]:
import os, json, ast, subprocess, tempfile, urllib.request, sys, shutil


# ---- locate the prepared vancomycin receptor that ships with this demo ----
_CANDIDATE_RECEPTORS = [
    "../examples/docking/vancomycin_receptor.pdbqt",   # ChemGraph repo
    "../data/vancomycin_receptor.pdbqt",               # dala-dock repo
    "data/vancomycin_receptor.pdbqt",
    "examples/docking/vancomycin_receptor.pdbqt",
    "vancomycin_receptor.pdbqt",
]


def find_vancomycin_receptor():
    for p in _CANDIDATE_RECEPTORS:
        if os.path.exists(p):
            return os.path.abspath(p)
    raise FileNotFoundError(
        "vancomycin_receptor.pdbqt not found next to this notebook. "
        "Prepare one with prepare_receptor('1FVM')."
    )


def prepare_receptor(source, out_prefix="receptor"):
    """Prepare a rigid receptor .pdbqt from a PDB ID, a .pdb file, or a SMILES.

    Requires Open Babel (conda install -c conda-forge openbabel). This is a *simple*
    preparation; for careful protein prep (chain selection, water removal) see
    prepare_receptor() in the dala-dock pipeline.
    """
    out = os.path.abspath(f"{out_prefix}.pdbqt")
    if isinstance(source, str) and source.lower().endswith(".pdbqt") and os.path.exists(source):
        return os.path.abspath(source)
    if os.path.exists(source) and source.lower().endswith(".pdb"):
        pdb = source
    elif isinstance(source, str) and len(source) == 4 and source.isalnum():
        pdb = f"{source}.pdb"
        urllib.request.urlretrieve(f"https://files.rcsb.org/download/{source}.pdb", pdb)
    else:
        # treat as a SMILES -> build a 3D rigid receptor
        subprocess.run(["obabel", f"-:{source}", "-O", out, "--gen3d", "-xr"], check=True)
        return out
    # protein PDB -> add hydrogens at pH 7.4, write a rigid receptor pdbqt
    subprocess.run(["obabel", pdb, "-O", out, "-xr", "-p", "7.4"], check=True)
    return out


def get_docking_result(state):
    """Pull the run_docking tool output (dict with poses_file + affinities) from the agent state."""
    msgs = state["messages"] if isinstance(state, dict) else getattr(state, "messages", [])
    for msg in reversed(msgs):
        name = getattr(msg, "name", None) or (msg.get("name") if isinstance(msg, dict) else None)
        content = getattr(msg, "content", None)
        if content is None and isinstance(msg, dict):
            content = msg.get("content")
        content = content if isinstance(content, str) else str(content)
        if name == "run_docking" or "poses_file" in content:
            for parse in (json.loads, ast.literal_eval):
                try:
                    d = parse(content)
                    if isinstance(d, dict) and d.get("poses_file"):
                        return d
                except Exception:
                    pass
    return None

def _obabel():
    exe = shutil.which("obabel") or os.path.join(os.path.dirname(sys.executable), "obabel")
    if not os.path.exists(exe):
        raise FileNotFoundError("Open Babel not found — conda install -c conda-forge openbabel")
    return exe


def _to_pdb(path, first_only=False):
    """Convert any Open Babel-readable file to PDB text (py3Dmol renders PDB reliably)."""
    cmd = ["obabel", path, "-opdb"] + (["-f", "1", "-l", "1"] if first_only else [])
    return subprocess.run(cmd, check=True, capture_output=True, text=True).stdout


def _split_models(poses_pdbqt):
    """Split a multi-pose file into individual PDB model strings (drops empty blocks)."""
    pdb = _to_pdb(poses_pdbqt)
    blocks = pdb.split("ENDMDL")
    return [b + "ENDMDL\n" for b in blocks if ("ATOM" in b or "HETATM" in b)]


def show_poses(receptor_pdbqt, poses_pdbqt, max_poses=5, width=720, height=520):
    """Interactive 3D view of the receptor (white) with the top docked poses (cyan)."""
    import py3Dmol

    view = py3Dmol.view(width=width, height=height)
    view.addModel(_to_pdb(receptor_pdbqt), "pdb")
    view.setStyle({"model": 0}, {"stick": {"colorscheme": "whiteCarbon", "radius": 0.12}})

    for i, model in enumerate(_split_models(poses_pdbqt)[:max_poses]):
        view.addModel(model, "pdb")
        view.setStyle({"model": i + 1}, {"stick": {"colorscheme": "cyanCarbon"}})

    view.zoomTo()
    return view.show()

def final_answer(state):
    """Agent's final text, whether messages are dicts or objects."""
    m = state["messages"][-1]
    return m["content"] if isinstance(m, dict) else m.content

print("helpers ready")


## Start the docking agent

`workflow_type="molecular_docking"` gives the agent the docking tools. `cg.visualize()`
prints the graph the agent runs.

In [ ]:
import os
from chemgraph.agent.llm_agent import ChemGraph

# Provide your API key via the shell (recommended):  export OPENAI_API_KEY="..."
# Or uncomment the next line and paste your key -- then DELETE it before committing.
# os.environ["OPENAI_API_KEY"] = "PASTE_YOUR_KEY_HERE"

cg = ChemGraph(
    model_name="openai/gpt-4o-mini",             # OpenRouter-namespaced id
    base_url="https://openrouter.ai/api/v1",     # OpenRouter endpoint
    workflow_type="molecular_docking",
    return_option="state",
)
print(cg.visualize())

## 1. Dock by just asking

The receptor here is the vancomycin target that ships with the demo. The **candidate** can
be a SMILES, a molecule name, or a PubChem CID — the agent resolves it for you.

In [ ]:
RECEPTOR = find_vancomycin_receptor()

query = (
    f"Dock aspirin into the receptor at '{RECEPTOR}' using 10 poses, "
    "and report the best binding affinity."
)
result = await cg.run(query, config={"configurable": {"thread_id": 1}})
print(final_answer(result))

### See the poses in 3D

In [ ]:
res = get_docking_result(result)
print("Best affinity:", res["best_affinity_kcal_mol"], "kcal/mol")
show_poses(RECEPTOR, res["poses_file"])

## 2. Any candidate — SMILES / name / PubChem CID

Same target, different candidate. Here we pass a **PubChem CID** to show name/CID resolution.

In [ ]:
query2 = f"Dock PubChem CID 3672 (ibuprofen) into the receptor at '{RECEPTOR}'."
result2 = await cg.run(query2, config={"configurable": {"thread_id": 2}})
print(final_answer(result2))

res2 = get_docking_result(result2)
show_poses(RECEPTOR, res2["poses_file"], max_poses=1)

## 3. Bring your own receptor (PDB ID / file / SMILES)

`prepare_receptor(...)` turns any of these into a docking-ready `.pdbqt`:
- a **4-letter PDB ID** (downloaded from RCSB),
- a local **`.pdb` file**,
- a **SMILES** (for a small-molecule target).

> This is a *simple* preparation (adds hydrogens, makes it rigid). For careful protein prep
> — selecting a chain, removing waters/ligands — use `prepare_receptor()` in
> `docking_demo.ipynb`, which does this more thoroughly.

In [ ]:
# pick ONE of these:
my_receptor = prepare_receptor("1FVM", out_prefix="my_target")     # a PDB ID
# my_receptor = prepare_receptor("path/to/target.pdb")             # a local PDB file
# my_receptor = prepare_receptor("c1ccccc1", out_prefix="benzene") # a SMILES

q = f"Dock ibuprofen into the receptor at '{my_receptor}'."
r = await cg.run(q, config={"configurable": {"thread_id": 3}})
print(final_answer(r))

show_poses(my_receptor, get_docking_result(r)["poses_file"])

## 4. Screen several candidates (with a progress bar)

For a whole panel we call the docking tool directly in a loop — this is much faster and
cheaper than one agent round-trip per molecule, and a `tqdm` bar shows how far along it is.
The result is a ranked leaderboard.

In [ ]:
from tqdm.auto import tqdm
import pandas as pd

from chemgraph.schemas.docking_schema import docking_input_schema
from chemgraph.tools.docking_core import run_docking_core

# name -> SMILES / molecule name / PubChem CID  (edit me!)
panel = {
    "aspirin": "aspirin",
    "ibuprofen": "ibuprofen",
    "acetaminophen": "acetaminophen",
    "caffeine": "2519",
}

rows = []
for name, cand in tqdm(panel.items(), desc="Docking"):
    out = run_docking_core(
        docking_input_schema(candidate=cand, receptor=RECEPTOR, n_poses=10)
    )
    rows.append({"candidate": name, "best_affinity_kcal_mol": out["best_affinity_kcal_mol"]})

board = (
    pd.DataFrame(rows)
    .sort_values("best_affinity_kcal_mol")
    .reset_index(drop=True)
)
board  # more negative = stronger predicted binding

In [ ]:
# scratch cell -- the notebook will PROMPT YOU to type a request when you run this
default_query = f"Dock caffeine into the receptor at '{RECEPTOR}'."
query = input("Enter your docking request (or press Enter for the default):\n> ") or default_query

result = await cg.run(query, config={"configurable": {"thread_id": 99}})
print(final_answer(result))
show_poses(RECEPTOR, get_docking_result(result)["poses_file"])

## 5. A text box to type your request (ipywidgets)

Prefer an input *field* over editing code? This gives you a text box and a **Run docking**
button — type any request and click to run it.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import asyncio, nest_asyncio
nest_asyncio.apply()

query_box = widgets.Text(
    value=f"Dock caffeine into the receptor at '{RECEPTOR}'.",
    placeholder="Type a docking request...",
    description="Query:",
    layout=widgets.Layout(width="90%"),
)
run_btn = widgets.Button(description="Run docking", button_style="primary")
out = widgets.Output()

async def _run_query():
    with out:
        out.clear_output()
        print("Running:", query_box.value)
        result = await cg.run(query_box.value, config={"configurable": {"thread_id": 42}})
        print(final_answer(result))
        show_poses(RECEPTOR, get_docking_result(result)["poses_file"])

run_btn.on_click(lambda _: asyncio.ensure_future(_run_query()))
display(query_box, run_btn, out)

## 6. Have a conversation (agent memory)

Reuse the **same `thread_id`** and the agent remembers the context, so you can refer back to
earlier steps in plain English.

In [ ]:
tid = {"configurable": {"thread_id": 7}}   # same thread across calls = the agent remembers

r1 = await cg.run(f"Dock aspirin into the receptor at '{RECEPTOR}'.", config=tid)
print("Q1:", final_answer(r1), "\n")

r2 = await cg.run("Now dock ibuprofen instead.", config=tid)
print("Q2:", final_answer(r2), "\n")

r3 = await cg.run("Which of the two bound more strongly, and why?", config=tid)
print("Q3:", final_answer(r3))

## 7. Look inside the agent

An AI agent decides *which tools to call* and *with what arguments*. This prints the agent's
step-by-step actions for any result — the tool calls it made and the messages in between.

In [ ]:
def _get(m, attr):
    v = getattr(m, attr, None)
    if v is None and isinstance(m, dict): v = m.get(attr)
    return v

def inspect_agent(state):
    """Show the agent's step-by-step actions: tool calls + messages."""
    for m in state["messages"]:
        role  = _get(m, "type") or _get(m, "role") or "?"
        name  = _get(m, "name")
        tools = _get(m, "tool_calls") or []
        if tools:
            for t in tools:
                tn = t.get("name") if isinstance(t, dict) else getattr(t, "name", "?")
                ta = t.get("args") if isinstance(t, dict) else getattr(t, "args", "")
                print(f"[{role}] -> calls tool: {tn}({ta})")
        else:
            c = _get(m, "content")
            c = c if isinstance(c, str) else str(c)
            label = role + (f"/{name}" if name else "")
            print(f"[{label}] {c.replace(chr(10), ' ')[:140]}")

inspect_agent(r3)   # <- pass any result from cg.run(...)

## 8. HPC & parallelization — screen many for the price of one

Virtual screening is **embarrassingly parallel**: each candidate docks completely
independently of the others. So:

- **Serial** (one at a time): total time  ≈  N × (one dock)
- **Parallel** (N workers at once): total time  ≈  (one dock)  *if you have enough compute*

That "if" is the whole point of HPC. A laptop has a handful of cores; a **Perlmutter CPU node
has 128 cores** (plus GPUs), and a **SLURM job array** spreads work across many nodes at once.
With enough workers, docking a 4-, 40-, or 400-molecule library takes about the same wall-clock
time as docking a **single** molecule — you just need the hardware to run them side by side.

Below we time the same `panel` (from Section 4) run **serially**, then **in parallel**, and
compare the wall time. (Run Section 4 first so `panel` is defined.)

### Serial — one molecule at a time (total ≈ N × one dock)

In [ ]:
%%time
serial = {}
for name, cand in panel.items():
    out = run_docking_core(docking_input_schema(candidate=cand, receptor=RECEPTOR, n_poses=10))
    serial[name] = round(out["best_affinity_kcal_mol"], 2)
print(serial)

### Parallel — all molecules at once (total ≈ one dock, if cores allow)

In [ ]:
%%time
from concurrent.futures import ThreadPoolExecutor
import os

def dock_one(item):
    name, cand = item
    out = run_docking_core(docking_input_schema(candidate=cand, receptor=RECEPTOR, n_poses=10))
    return name, round(out["best_affinity_kcal_mol"], 2)

# one worker per candidate (capped by available cores)
workers = min(len(panel), os.cpu_count() or 1)
with ThreadPoolExecutor(max_workers=workers) as ex:
    parallel = dict(ex.map(dock_one, panel.items()))
print(f"ran {len(panel)} docks across {workers} workers")
print(parallel)

### What you should see

Both runs give the **same scores** — but the parallel run's **wall time** is much shorter,
ideally approaching a *single* dock (capped by how many cores you have). On a laptop the
speedup is modest (few cores, and Vina already uses several threads per dock); on an HPC node
it is dramatic.

**Scaling further on real HPC:**
- **More cores / a bigger node** → more docks running at once.
- **SLURM job arrays** → one dock per array task, spread across many nodes simultaneously.
- **AutoDock-GPU** → the GPU batches many ligands in a single run — parallelism in the hardware itself.

**Takeaway:** on HPC the cost of screening a library is set by your *throughput* (how many
workers you can run), **not** by the number of molecules. That is why HPC transforms virtual
screening — and why "dock 400 candidates" can cost about the same wall-clock time as "dock 1".